# 03 — Exportación de datos para Power BI
## easyMoney | TFM Data Science & AI — Nuclio School

**Prerequisitos:** `01-eda.ipynb` y `02-eda-deep-dive.ipynb` ejecutados

**Objetivo:** Exportar los datos limpios (sin anomalías) en formato CSV
para construir el dashboard de BI en Power BI Desktop.

**Input:** `master_df_flags.parquet` — tabla maestra con flags de calidad

**Outputs:**
- `period_summary.csv` — evolución temporal de KPIs
- `product_penetration.csv` — penetración por producto y período
- `client_profile.csv` — perfil demográfico Mayo 2019
- `kpi_summary.csv` — KPIs calculados por período
- `product_penetration_long.csv` — penetración por producto en formato largo para gráfico de barras

---

In [1]:
# ── 03 — Exportación de datos para Power BI ───────────────────────────────
import pandas as pd
import os

BASE = 'C:\\Users\\farno\\OneDrive\\Desktop\\Data science & AI - Nuclio School\\proyecto final TFM\\tfm-fintech-easymoney\\data\\processed\\'

DATA_PATH   = BASE + 'master_df_flags.parquet'
OUTPUT_PATH = BASE + 'powerbi\\'

df = pd.read_parquet(DATA_PATH)
os.makedirs(OUTPUT_PATH, exist_ok=True)

product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                'long_term_deposit','credit_card','payroll','pension_plan',
                'payroll_account','emc_account','debit_card','em_account_p',
                'em_acount']

last_partition = df['pk_partition'].max()

# ── Verificación ───────────────────────────────────────────────────────────
print(f"✓ Parquet cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"✓ Flags: {[c for c in df.columns if 'anomaly' in c]}")
print(f"✓ Última partición: {last_partition}")

✓ Parquet cargado: 5,962,924 filas × 37 columnas
✓ Flags: ['age_anomaly', 'deceased_anomaly', 'entry_date_anomaly']
✓ Última partición: 2019-05-28 00:00:00


In [2]:
# ── Exportar CSVs para Power BI ───────────────────────────────────────────

# Tabla 1: Resumen por período
period_summary_export = df.groupby('pk_partition').agg(
    total_clients        = ('pk_cid', 'count'),
    active_clients       = ('active_customer', 'sum'),
    new_clients          = ('is_new_client', 'sum'),
    new_contracts        = ('new_contracts', 'sum'),
    avg_products         = ('total_products', 'mean'),
    clients_0_products   = ('total_products', lambda x: (x==0).sum()),
    clients_1_product    = ('total_products', lambda x: (x==1).sum()),
    clients_2plus        = ('total_products', lambda x: (x>=2).sum()),
).reset_index()
period_summary_export['pk_partition'] = period_summary_export['pk_partition'].astype(str).str[:10]
period_summary_export.to_csv(OUTPUT_PATH + 'period_summary.csv', index=False)
print(f"✓ period_summary.csv — {period_summary_export.shape}")

# Tabla 2: Penetración por producto por período
product_penetration = df.groupby('pk_partition')[product_cols].mean().mul(100).round(2).reset_index()
product_penetration['pk_partition'] = product_penetration['pk_partition'].astype(str).str[:10]
product_penetration.to_csv(OUTPUT_PATH + 'product_penetration.csv', index=False)
print(f"✓ product_penetration.csv — {product_penetration.shape}")

# Tabla 3: Perfil cliente última partición
df_clean = df[~df[['age_anomaly','deceased_anomaly','entry_date_anomaly']].any(axis=1)]
client_profile = df_clean[df_clean['pk_partition'] == last_partition][[
    'pk_cid', 'segment', 'age_group', 'salary_group',
    'gender', 'region_code', 'country_id',
    'total_products', 'is_new_client',
    'client_age_months', 'active_customer'
] + product_cols].copy()
client_profile['pk_partition'] = str(last_partition)[:10]
client_profile.to_csv(OUTPUT_PATH + 'client_profile.csv', index=False)
print(f"✓ client_profile.csv — {client_profile.shape}")

# Tabla 4: KPIs resumen
kpi_summary = period_summary_export.copy()
kpi_summary['pct_new_clients'] = (kpi_summary['new_clients'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_0_products']  = (kpi_summary['clients_0_products'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_1_product']   = (kpi_summary['clients_1_product'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary['pct_crosssell']   = (kpi_summary['clients_2plus'] / kpi_summary['total_clients'] * 100).round(2)
kpi_summary.to_csv(OUTPUT_PATH + 'kpi_summary.csv', index=False)
print(f"✓ kpi_summary.csv — {kpi_summary.shape}")

print(f"\n✓ Todos los archivos exportados en: {OUTPUT_PATH}")
print(f"\nArchivos Power BI:")
for f in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH + f) / 1024
    print(f"  {f:<35} {size:.1f} KB")

✓ period_summary.csv — (17, 9)
✓ product_penetration.csv — (17, 15)
✓ client_profile.csv — (442157, 26)
✓ kpi_summary.csv — (17, 13)

✓ Todos los archivos exportados en: C:\Users\farno\OneDrive\Desktop\Data science & AI - Nuclio School\proyecto final TFM\tfm-fintech-easymoney\data\processed\powerbi\

Archivos Power BI:
  client_profile.csv                  42413.5 KB
  kpi_summary.csv                     1.8 KB
  period_summary.csv                  1.4 KB
  product_penetration.csv             1.5 KB


In [4]:
# ── Tabla 5: Penetración por producto (formato largo para Power BI) ────────

label_map = {
    'em_acount': 'Cuenta easyMoney',
    'payroll': 'Domiciliaciones',
    'em_account_p': 'Cuenta easyMoney+',
    'debit_card': 'Tarjeta débito',
    'credit_card': 'Tarjeta crédito',
    'payroll_account': 'Cuenta nómina',
    'emc_account': 'Cuenta Crypto',
    'short_term_deposit': 'Depósito C/P',
    'long_term_deposit': 'Depósito L/P',
    'pension_plan': 'Plan pensiones',
    'funds': 'Fondos inversión',
    'securities': 'Valores',
    'mortgage': 'Hipoteca',
    'loans': 'Préstamos'
}

df_last = df[df['pk_partition'] == last_partition]

penetration_long = pd.DataFrame({
    'producto': list(label_map.values()),
    'nombre_tecnico': list(label_map.keys()),
    'penetracion_pct': [df_last[col].mean() * 100 for col in label_map.keys()]
}).round(2).sort_values('penetracion_pct', ascending=False)

penetration_long.to_csv(OUTPUT_PATH + 'product_penetration_long.csv', index=False)

print(f"✓ product_penetration_long.csv — {penetration_long.shape}")
print(f"\nVista previa:")
print(penetration_long.to_string(index=False))

✓ product_penetration_long.csv — (14, 3)

Vista previa:
         producto     nombre_tecnico  penetracion_pct
 Cuenta easyMoney          em_acount            66.90
   Tarjeta débito         debit_card             9.77
    Cuenta nómina    payroll_account             5.99
    Cuenta Crypto        emc_account             5.59
   Plan pensiones       pension_plan             3.92
  Domiciliaciones            payroll             3.69
     Depósito L/P  long_term_deposit             1.38
  Tarjeta crédito        credit_card             1.08
          Valores         securities             0.40
 Fondos inversión              funds             0.30
        Préstamos              loans             0.01
         Hipoteca           mortgage             0.01
Cuenta easyMoney+       em_account_p             0.00
     Depósito C/P short_term_deposit             0.00
